# Retrieval Layer 

Current pipeline:
- **Milvus Lite** retrieval from `milvus_bge_m3.db`
- **BGE-M3 dense query encoding** via Hugging Face `transformers`
- **Final chunks** as the input corpus
- **Schema-grounded gating** using `cco schema.json`
- **Pattern-aware ODP retrieval** using `Ontology design patterns.json`

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade "transformers>=4.41,<5.0.0" sentencepiece accelerate "pymilvus[milvus_lite]"
print("Installed packages for retrieval layer (Milvus Lite + transformers).")


In [ ]:
import os
import shutil

# -- Paths -------------------------------------------------------------------
MILVUS_URI = "../1 - Data Preparation Layer/Embdedding Generation/output/milvus_bge_m3.db"
PREPARED_UNITS_DIR = "../1 - Data Preparation Layer/PDFs Processing/output/preparation_v1"
PREPARED_UNIT_FILES = {
    "UK": "final_chunks_uk_v1.json",
    "Canada": "final_chunks_canada_v1.json",
    "Australia": "final_chunks_australia_v1.json",
    "USA": "final_chunks_usa_v1.json",
}

OUTPUT_DIR = "output/retrieval_contexts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MILVUS_WORKING_COPY = os.path.join(OUTPUT_DIR, "milvus_bge_m3_retrieval.db")

# -- Collections -------------------------------------------------------------
SCHEMA_COLLECTION = "cco_Schema"
ODP_COLLECTION = "ODP"
UNITS_COLLECTION = "final_chunksv1"

# -- Query encoder -----------------------------------------------------------
EMBEDDING_MODEL = "BAAI/bge-m3"
MAX_LENGTH = 1024
BATCH_SIZE = 16

# -- Retrieval config (simplified: 2 thresholds, was 10) ---------------------
SCHEMA_TOP_K = 8
ODP_TOP_K = 5

SCHEMA_MIN_SCORE = 0.60
ODP_MIN_SCORE = 0.60
MIN_QUERY_WORDS = 10

MAX_SCHEMA_RESULTS = 5
MAX_PATTERN_RESULTS = 3

# Generic labels that score high but are too abstract for downstream use
GENERIC_SCHEMA_LABELS = {
    "Agent", "Person", "Organisation", "Role", "Role Holding", "RoleHolding",
    "Regulation", "Norm", "Regulatory Authority Agent", "Regulatory Authority Role",
}

# Schema labels signalling temporal data properties (used for temporal signal detection)
TEMPORAL_SCHEMA_TERMS = {
    "hasApplicabilityStart", "hasApplicabilityEnd",
    "hasValidityStart", "hasValidityEnd",
    "hasStartTime", "hasEndTime",
    "has applicability start", "has applicability end",
    "has validity start", "has validity end",
    "has start time", "has end time",
}

print("Configuration (v26 simplified):")
print(f"  Milvus DB           : {MILVUS_URI}  exists={os.path.exists(MILVUS_URI)}")
print(f"  Prepared units dir  : {PREPARED_UNITS_DIR}")
print(f"  Milvus working copy : {MILVUS_WORKING_COPY}")
print(f"  Embedding model     : {EMBEDDING_MODEL}")
print(f"  Schema collection   : {SCHEMA_COLLECTION}")
print(f"  ODP collection      : {ODP_COLLECTION}")
print(f"  Units collection    : {UNITS_COLLECTION}")
print(f"  Schema top-k        : {SCHEMA_TOP_K}")
print(f"  ODP top-k           : {ODP_TOP_K}")
print(f"  Schema min score    : {SCHEMA_MIN_SCORE}")
print(f"  ODP min score       : {ODP_MIN_SCORE}")
print(f"  Min query words     : {MIN_QUERY_WORDS}")
for jurisdiction, filename in PREPARED_UNIT_FILES.items():
    path = os.path.join(PREPARED_UNITS_DIR, filename)
    print(f"  {jurisdiction:<10}      : exists={os.path.exists(path)}")


In [ ]:
import json
import re
from datetime import datetime
from typing import Any, Dict, List

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from pymilvus import Collection, connections

print("Imports OK")


In [ ]:
# -- Load query embedding model -----------------------------------------------
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model: {EMBEDDING_MODEL} on {DEVICE} ...")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
model = AutoModel.from_pretrained(EMBEDDING_MODEL)
model.to(DEVICE)
model.eval()

with torch.no_grad():
    probe_inputs = tokenizer(
        ["probe text"],
        padding=True,
        truncation=True,
        max_length=16,
        return_tensors="pt",
    )
    probe_inputs = {k: v.to(DEVICE) for k, v in probe_inputs.items()}
    probe_outputs = model(**probe_inputs)
    probe_vec = F.normalize(probe_outputs.last_hidden_state[:, 0], p=2, dim=1)
    DENSE_DIM = probe_vec.shape[1]
print(f"  Model loaded. Dense dim: {DENSE_DIM}")

if not os.path.exists(MILVUS_URI):
    raise FileNotFoundError(f"Milvus DB not found: {MILVUS_URI}")

try:
    if os.path.exists(MILVUS_WORKING_COPY):
        os.remove(MILVUS_WORKING_COPY)
    shutil.copy2(MILVUS_URI, MILVUS_WORKING_COPY)
    print(f"Copied Milvus DB to working copy: {MILVUS_WORKING_COPY}")
except Exception as exc:
    raise RuntimeError(f"Failed to prepare Milvus working copy: {exc}") from exc

connections.connect(alias="default", uri=MILVUS_WORKING_COPY)
collection_schema = Collection(SCHEMA_COLLECTION)
collection_odp = Collection(ODP_COLLECTION)
collection_units = Collection(UNITS_COLLECTION)
collection_schema.load()
collection_odp.load()
collection_units.load()

print("Collections loaded:")
print(f"  {SCHEMA_COLLECTION}: {collection_schema.num_entities}")
print(f"  {ODP_COLLECTION}: {collection_odp.num_entities}")
print(f"  {UNITS_COLLECTION}: {collection_units.num_entities}")


In [ ]:
def embed_dense(texts: List[str], batch_size: int = BATCH_SIZE) -> List[List[float]]:
    """Generate dense BGE-M3 embeddings using normalized [CLS] states."""
    vectors: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            batch_vecs = F.normalize(outputs.last_hidden_state[:, 0], p=2, dim=1)
        vectors.extend(batch_vecs.cpu().tolist())
    return vectors


def milvus_search(
    collection: Collection,
    query_vector: List[float],
    limit: int,
    output_fields: List[str],
) -> List[Dict[str, Any]]:
    """Run a single COSINE search against a Milvus collection."""
    results = collection.search(
        data=[query_vector],
        anns_field="dense_vector",
        param={"metric_type": "COSINE", "params": {}},
        limit=limit,
        output_fields=output_fields,
    )
    hits: List[Dict[str, Any]] = []
    for hit in results[0]:
        entity = hit.entity
        row: Dict[str, Any] = {}
        for field in output_fields:
            try:
                row[field] = entity.get(field)
            except Exception:
                row[field] = None
        row["score"] = float(getattr(hit, "score", getattr(hit, "distance", 0.0)))
        hits.append(row)
    return hits


def load_prepared_units() -> Dict[str, List[Dict[str, Any]]]:
    """Load prepared units for each jurisdiction."""
    prepared: Dict[str, List[Dict[str, Any]]] = {}
    for jurisdiction, filename in PREPARED_UNIT_FILES.items():
        path = os.path.join(PREPARED_UNITS_DIR, filename)
        with open(path, encoding="utf-8") as f:
            payload = json.load(f)
        prepared[jurisdiction] = payload["final_chunks"]
    return prepared


def build_unit_query_text(unit: Dict[str, Any]) -> str:
    """Construct query text from a prepared unit."""
    return (unit.get("text") or "").strip()


def normalized_label(hit: Dict[str, Any]) -> str:
    return (hit.get("label") or hit.get("local_name") or "").strip()


def informative_primary_hits(schema_hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Filter schema hits to classes, object properties, and data properties,
    excluding over-generic labels. Data properties are included because they
    signal temporal and expression content (e.g., hasApplicabilityStart,
    hasConditionExpression) that is regulatory content even when no class
    matches strongly."""
    informative = []
    for hit in schema_hits:
        if hit.get("item_type") not in {"class", "object_property", "data_property"}:
            continue
        if normalized_label(hit) in GENERIC_SCHEMA_LABELS:
            continue
        informative.append(hit)
    return informative


# -----------------------------------------------------------------------------
# Schema-based signal detection (replaces regex-based detection in v25)
# All signals are derived from Stage 1 schema gate hits, not from raw text.
# -----------------------------------------------------------------------------

def has_funding_signal(schema_gate: Dict[str, Any]) -> bool:
    """True if schema gate found resource/allocation-related entities."""
    labels = {normalized_label(hit) for hit in schema_gate["filtered_schema_hits"]}
    return bool(labels & {"Resource", "allocated to", "has object"})


def has_authority_signal(schema_gate: Dict[str, Any]) -> bool:
    """True if schema gate found regulatory-authority-related entities."""
    labels = {normalized_label(hit) for hit in schema_gate["filtered_schema_hits"]}
    return bool(labels & {
        "Regulatory Authority Agent", "Regulatory Authority Role", "issues",
    })


def has_temporal_signal(schema_gate: Dict[str, Any]) -> bool:
    """True if schema gate found temporal data properties."""
    for hit in schema_gate["filtered_schema_hits"]:
        if hit.get("local_name") in TEMPORAL_SCHEMA_TERMS:
            return True
        if normalized_label(hit) in TEMPORAL_SCHEMA_TERMS:
            return True
    return False


# -----------------------------------------------------------------------------
# Stage 1: Schema-grounded candidate gating (simplified)
# -----------------------------------------------------------------------------

def schema_grounded_candidate_gating(
    unit: Dict[str, Any], schema_hits: List[Dict[str, Any]]
) -> Dict[str, Any]:
    """Single-threshold schema gate.

    Accept a chunk if at least one informative schema hit (class or object
    property, excluding over-generic labels) scores >= SCHEMA_MIN_SCORE.
    """
    filtered = [hit for hit in schema_hits if hit["score"] >= SCHEMA_MIN_SCORE]
    informative = informative_primary_hits(filtered)

    selected = bool(informative)
    reason = "informative_schema_match" if selected else "no_informative_schema_match"

    return {
        "selected": selected,
        "reason": reason,
        "filtered_schema_hits": filtered[:MAX_SCHEMA_RESULTS],
        "informative_schema_hits": informative[:MAX_SCHEMA_RESULTS],
    }


# -----------------------------------------------------------------------------
# Stage 2: Pattern-aware retrieval (simplified)
# Pattern-specific penalties are retained, but signal sources are now the
# Stage 1 schema gate hits rather than regex over raw chunk text.
# -----------------------------------------------------------------------------

def pattern_aware_adaptive_retrieval(
    unit: Dict[str, Any],
    query_vector: List[float],
    schema_gate: Dict[str, Any],
) -> Dict[str, Any]:
    """Single-threshold ODP gate with schema-grounded penalty signals."""
    raw_hits = milvus_search(
        collection_odp,
        query_vector,
        limit=ODP_TOP_K,
        output_fields=["item_id", "label", "category", "pipeline_layer", "text"],
    )

    funding_signal = has_funding_signal(schema_gate)
    authority_signal = has_authority_signal(schema_gate)
    temporal_signal = has_temporal_signal(schema_gate)

    ranked: List[Dict[str, Any]] = []
    for hit in raw_hits:
        label = hit.get("label") or ""
        adjusted = hit["score"]

        if label == "Funding Allocation Pattern" and not funding_signal:
            adjusted -= 0.20
        if label == "Regulatory Authority Pattern" and not authority_signal:
            adjusted -= 0.08
        if label == "Norm Temporal Applicability Pattern" and not temporal_signal:
            adjusted -= 0.08

        if adjusted >= ODP_MIN_SCORE:
            ranked.append({**hit, "score": round(adjusted, 4)})

    ranked.sort(key=lambda item: item["score"], reverse=True)
    top_hits = ranked[:MAX_PATTERN_RESULTS]

    selected = bool(top_hits)
    reason = "pattern_match" if selected else "no_pattern_match"

    return {
        "selected": selected,
        "reason": reason,
        "top_hits": top_hits,
    }


# -----------------------------------------------------------------------------
# Top-level: retrieve for a single unit
# -----------------------------------------------------------------------------

def retrieve_for_unit(unit: Dict[str, Any]) -> Dict[str, Any]:
    query_text = build_unit_query_text(unit)

    # Short-chunk filter: list fragments and headings (< MIN_QUERY_WORDS words)
    # are rejected before retrieval. These are typically not substantive
    # regulatory content and produce noisy LLM extraction.
    if len(query_text.split()) < MIN_QUERY_WORDS:
        return {
            "unit_id": unit.get("unit_id"),
            "jurisdiction": unit.get("jurisdiction"),
            "unit_type": unit.get("unit_type"),
            "section": unit.get("section"),
            "section_path": unit.get("section_path") or [],
            "local_heading": unit.get("local_heading"),
            "source_anchor": unit.get("source_anchor"),
            "text": unit.get("text", ""),
            "query_text": query_text,
            "selected_for_llm": False,
            "schema_gate": {"selected": False, "reason": "short_chunk_filtered"},
            "odp_gate": {"selected": False, "reason": "short_chunk_filtered"},
            "schema_hits": [],
            "odp_hits": [],
        }

    query_vector = embed_dense([query_text], batch_size=1)[0]

    schema_hits = milvus_search(
        collection_schema,
        query_vector,
        limit=SCHEMA_TOP_K,
        output_fields=["item_id", "item_type", "label", "local_name", "text"],
    )
    schema_gate = schema_grounded_candidate_gating(unit, schema_hits)

    odp_gate = {"selected": False, "reason": "schema_rejected"}
    odp_hits: List[Dict[str, Any]] = []
    if schema_gate["selected"]:
        odp_stage = pattern_aware_adaptive_retrieval(unit, query_vector, schema_gate)
        odp_gate = {
            "selected": odp_stage["selected"],
            "reason": odp_stage["reason"],
        }
        if odp_stage["selected"]:
            odp_hits = odp_stage["top_hits"]

    final_selected = schema_gate["selected"] and odp_gate["selected"]

    return {
        "unit_id": unit.get("unit_id"),
        "jurisdiction": unit.get("jurisdiction"),
        "unit_type": unit.get("unit_type"),
        "section": unit.get("section"),
        "section_path": unit.get("section_path") or [],
        "local_heading": unit.get("local_heading"),
        "source_anchor": unit.get("source_anchor"),
        "text": unit.get("text", ""),
        "query_text": query_text,
        "selected_for_llm": final_selected,
        "schema_gate": {
            "selected": schema_gate["selected"],
            "reason": schema_gate["reason"],
        },
        "odp_gate": odp_gate,
        "schema_hits": schema_gate["filtered_schema_hits"],
        "odp_hits": odp_hits,
    }


print("Helper functions defined ( data props included, thresholds raised, short filter).")


In [ ]:
prepared_units_by_jurisdiction = load_prepared_units()
print("Loaded prepared units:")
for jurisdiction, units in prepared_units_by_jurisdiction.items():
    print(f"  {jurisdiction}: {len(units)} units")


In [ ]:
TEST_UNITS = [
    prepared_units_by_jurisdiction["UK"][79],
    prepared_units_by_jurisdiction["Canada"][120],
    prepared_units_by_jurisdiction["Australia"][35],
    prepared_units_by_jurisdiction["USA"][40],
]

print("=" * 70)
print("RETRIEVAL TEST - 4 sample prepared units (v25)")
print("=" * 70)
for unit in TEST_UNITS:
    result = retrieve_for_unit(unit)
    print("\n" + "-" * 70)
    print(f"Unit: [{result['unit_id']}] [{result['unit_type']}] [{result['jurisdiction']}]")
    print(f"Section: {result['section']}")
    print(f"Text   : {result['text'][:160]}...")
    print(f"\nSchema gate: {result['schema_gate']['selected']} ({result['schema_gate']['reason']})")
    print(f"Schema hits ({len(result['schema_hits'])}):")
    for hit in result['schema_hits']:
        print(f"  [{hit['item_id']}] {hit['label']:<32} score={hit['score']:.4f} ({hit['item_type']})")
    print(f"ODP gate  : {result['odp_gate']['selected']} ({result['odp_gate']['reason']})")
    print(f"\nODP hits ({len(result['odp_hits'])}):")
    for hit in result['odp_hits']:
        print(f"  [{hit['item_id']}] {hit['label']:<36} score={hit['score']:.4f} (cat={hit['category']})")


In [ ]:
all_retrieval_contexts = {}

for jurisdiction, units in prepared_units_by_jurisdiction.items():
    print(f"\nProcessing {jurisdiction}: {len(units)} prepared units...")
    retrieval_contexts = []
    selected_count = 0
    schema_selected_count = 0
    empty_schema = 0
    empty_odp = 0

    for i, unit in enumerate(units):
        ctx = retrieve_for_unit(unit)
        retrieval_contexts.append(ctx)
        if ctx["schema_gate"]["selected"]:
            schema_selected_count += 1
        if ctx["selected_for_llm"]:
            selected_count += 1
        if not ctx["schema_hits"]:
            empty_schema += 1
        if not ctx["odp_hits"]:
            empty_odp += 1
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(units)} done...")

    out_path = os.path.join(OUTPUT_DIR, f"retrieval_{jurisdiction.lower()}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "metadata": {
                    "jurisdiction": jurisdiction,
                    "total_units": len(units),
                    "selected_after_schema_gate": schema_selected_count,
                    "selected_for_llm": selected_count,
                    "empty_schema_hits": empty_schema,
                    "empty_odp_hits": empty_odp,
                    "schema_top_k": SCHEMA_TOP_K,
                    "odp_top_k": ODP_TOP_K,
                    "schema_min_score": SCHEMA_MIN_SCORE,
                        "odp_min_score": ODP_MIN_SCORE,
                "min_query_words": MIN_QUERY_WORDS,
                    "embedding_model": EMBEDDING_MODEL,
                    "vector_store": "Milvus Lite",
                    "pipeline_design": "Schema-grounded candidate gating + pattern-aware adaptive retrieval",
                    "created_at": datetime.now().isoformat(),
                },
                "retrieval_contexts": retrieval_contexts,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    all_retrieval_contexts[jurisdiction] = retrieval_contexts
    print(f"  Saved: {out_path}")
    print(f"  Selected after schema gate: {schema_selected_count}/{len(units)}")
    print(f"  Selected for LLM         : {selected_count}/{len(units)}")
    print(f"  Empty schema hits         : {empty_schema}")
    print(f"  Empty ODP hits            : {empty_odp}")


In [ ]:
sample = all_retrieval_contexts["UK"][30]
print(f"Unit ID      : {sample['unit_id']}")
print(f"Jurisdiction : {sample['jurisdiction']}")
print(f"Unit type    : {sample['unit_type']}")
print(f"Section      : {sample['section']}")
print(f"Text         : {sample['text'][:160]}...")
print(f"\nSelected for LLM : {sample['selected_for_llm']}")
print(f"Schema gate      : {sample['schema_gate']}")
print(f"ODP gate         : {sample['odp_gate']}")
print(f"\nSchema hits ({len(sample['schema_hits'])}):")
for hit in sample['schema_hits']:
    print(f"  [{hit['item_id']}] {hit['label']:<32} score={hit['score']:.4f} ({hit['item_type']})")
print(f"\nODP hits ({len(sample['odp_hits'])}):")
for hit in sample['odp_hits']:
    print(f"  [{hit['item_id']}] {hit['label']:<36} score={hit['score']:.4f} (cat={hit['category']})")
